# Day 5: Multi-Agent RL, Advanced Topics & Capstone

**Reinforcement Learning: From Theory to Practice**

*Dr. Yaé Ulrich Gaba — AIRINA Labs / ACAS*

---

## Overview

Welcome to the final day of the workshop. Today we close the loop from single-agent fundamentals to real-world deployment:

1. **Multi-Agent RL (MARL)** — cooperative vs. competitive settings, independent learners
2. **Self-Play** — training agents by competing against earlier versions of themselves
3. **Model-Based RL** — world models and Dyna-Q for sample efficiency
4. **Offline RL & Imitation Learning** — learning from fixed datasets and human demonstrations
5. **Capstone Project** — full RL pipeline on a custom African energy-grid resource-allocation environment
6. **Workshop Summary** — five-day recap, research frontiers, recommended reading

---

### Prerequisites
Days 1–4: MDPs, tabular methods (Q-learning, SARSA), DQN, policy gradients (REINFORCE, A2C, PPO).

### Library requirements
```
pip install numpy matplotlib gymnasium stable-baselines3 torch
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict, deque
import random
import warnings
warnings.filterwarnings('ignore')

import gymnasium as gym
from gymnasium import spaces

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.distributions import Categorical

# Stable-Baselines3 (optional — graceful fallback if absent)
try:
    from stable_baselines3 import PPO, DQN
    from stable_baselines3.common.evaluation import evaluate_policy
    from stable_baselines3.common.callbacks import BaseCallback
    SB3_AVAILABLE = True
    print('Stable-Baselines3 loaded.')
except ImportError:
    SB3_AVAILABLE = False
    print('stable-baselines3 not found — SB3 cells will be skipped.')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

# Global plot style
%matplotlib inline
plt.rcParams['figure.figsize'] = (11, 5)
plt.rcParams['font.size'] = 12
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
random.seed(SEED)

---
## Part 1 — Multi-Agent Reinforcement Learning (MARL)

### 1.1 From Single-Agent to Multi-Agent Settings

In single-agent RL the environment is **stationary** from the learner's perspective: the transition and reward functions do not depend on other decision-makers. In multi-agent systems this assumption breaks down.

**Formal definition.** A Markov Game (Stochastic Game) for $N$ agents is the tuple

$$\mathcal{M} = \langle \mathcal{S},\, \{\mathcal{A}^i\}_{i=1}^N,\, P,\, \{r^i\}_{i=1}^N,\, \gamma \rangle$$

where the joint action $\mathbf{a} = (a^1, \dots, a^N)$ determines the transition $P(s' | s, \mathbf{a})$ and each agent $i$ receives its own reward $r^i(s, \mathbf{a}, s')$.

### 1.2 Taxonomy of Multi-Agent Problems

| Setting | Reward structure | Example |
|---------|------------------|---------|
| **Fully cooperative** | $r^1 = r^2 = \cdots = r^N$ | Warehouse robots, sensor networks |
| **Fully competitive** (zero-sum) | $\sum_i r^i = 0$ | Chess, Poker, Go |
| **Mixed / general-sum** | arbitrary | Auctions, traffic, energy markets |

### 1.3 Key Challenges

1. **Non-stationarity** — from agent $i$'s viewpoint, other agents are part of a changing environment.
2. **Credit assignment** — in cooperative settings, who contributed to a shared reward?
3. **Scalability** — joint action space grows exponentially: $|\mathcal{A}|^N$.
4. **Equilibrium selection** — multiple Nash equilibria may exist.

### 1.4 Independent Q-Learning (IQL)

The simplest baseline: each agent applies single-agent Q-learning, treating other agents as part of the environment. Despite violating the stationarity assumption it often works in practice.

In [ ]:
# ---------------------------------------------------------------
# Cooperative Grid-World (shared reward)
# Two agents on a 5x5 grid must both reach their target cells.
# State  : (row1, col1, row2, col2)
# Actions: 0=up 1=down 2=left 3=right  (per agent)
# Reward : +1 shared when BOTH agents are on their targets simultaneously
# ---------------------------------------------------------------

GRID = 5
TARGETS = [(0, 4), (4, 0)]   # target for agent 0 and agent 1
N_ACTIONS = 4
DELTAS = [(-1, 0), (1, 0), (0, -1), (0, 1)]  # up, down, left, right


def step_agent(pos, action):
    """Move agent from pos=(r,c) with action; clip to grid boundaries."""
    dr, dc = DELTAS[action]
    return (min(max(pos[0] + dr, 0), GRID - 1),
            min(max(pos[1] + dc, 0), GRID - 1))


class CoopGridWorld:
    """Minimal cooperative two-agent grid world."""

    def reset(self, seed=None):
        rng = np.random.default_rng(seed)
        # Random start positions that are not the targets
        all_cells = [(r, c) for r in range(GRID) for c in range(GRID)
                     if (r, c) not in TARGETS]
        starts = rng.choice(len(all_cells), size=2, replace=False)
        self.pos = [all_cells[starts[0]], all_cells[starts[1]]]
        self.steps = 0
        return tuple(self.pos[0] + self.pos[1])

    def step(self, actions):
        self.pos[0] = step_agent(self.pos[0], actions[0])
        self.pos[1] = step_agent(self.pos[1], actions[1])
        self.steps += 1
        both_done = (self.pos[0] == TARGETS[0] and self.pos[1] == TARGETS[1])
        reward = 1.0 if both_done else -0.01   # small penalty per step
        done = both_done or self.steps >= 100
        state = tuple(list(self.pos[0]) + list(self.pos[1]))
        return state, reward, done


# Independent Q-Learning
def iql_coop(n_episodes=3000, alpha=0.1, gamma=0.95,
             eps_start=1.0, eps_end=0.05, eps_decay=0.999):
    """Train two agents with Independent Q-Learning on CoopGridWorld."""
    env = CoopGridWorld()
    # Q-tables indexed by (state, action) — use defaultdict for sparse storage
    Q = [defaultdict(float), defaultdict(float)]
    eps = eps_start
    ep_rewards = []
    ep_lengths = []

    for ep in range(n_episodes):
        state = env.reset(seed=ep)
        total_r = 0.0
        done = False

        while not done:
            actions = []
            for i in range(2):
                if random.random() < eps:
                    actions.append(random.randrange(N_ACTIONS))
                else:
                    q_vals = [Q[i][(state, a)] for a in range(N_ACTIONS)]
                    actions.append(int(np.argmax(q_vals)))

            next_state, reward, done = env.step(actions)
            total_r += reward

            for i in range(2):
                best_next = max(Q[i][(next_state, a)] for a in range(N_ACTIONS))
                td_target = reward + gamma * best_next * (not done)
                Q[i][(state, actions[i])] += alpha * (
                    td_target - Q[i][(state, actions[i])]
                )

            state = next_state

        eps = max(eps_end, eps * eps_decay)
        ep_rewards.append(total_r)
        ep_lengths.append(env.steps)

    return ep_rewards, ep_lengths, Q


print('Training IQL on CoopGridWorld...')
iql_rewards, iql_lengths, Q_iql = iql_coop(n_episodes=3000)
print(f'Final 200-ep avg reward : {np.mean(iql_rewards[-200:]):.4f}')
print(f'Final 200-ep avg length : {np.mean(iql_lengths[-200:]):.1f} steps')

In [ ]:
window = 100

def smooth(arr, w):
    return np.convolve(arr, np.ones(w) / w, mode='valid')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(smooth(iql_rewards, window), color='#1565C0', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Total Reward')
ax.set_title('IQL — Cooperative Grid World\n(Episode Reward)')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(smooth(iql_lengths, window), color='#B71C1C', linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Steps to Complete')
ax.set_title('IQL — Episode Length')
ax.grid(True, alpha=0.3)

plt.suptitle('Independent Q-Learning — Two Cooperative Agents', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

print('\nNote: reward increases and episode length decreases as agents learn.\n'
      'Despite non-stationarity, IQL converges reasonably on small cooperative tasks.')

---
## Part 2 — Self-Play

### 2.1 Motivation

In two-player zero-sum games, the ideal training partner is the **current best opponent** — but we do not have that at the start. **Self-play** bootstraps this:

1. Initialise agent $\pi_0$ randomly.
2. At each training iteration, pit $\pi_\theta$ against a frozen copy $\pi_{\theta'}$ (the *opponent*).
3. Update $\pi_\theta$ to beat $\pi_{\theta'}$, then set $\pi_{\theta'} \leftarrow \pi_\theta$.

Key results using self-play: **AlphaGo Zero**, **OpenAI Five (Dota)**, **AlphaStar (StarCraft II)**.

### 2.2 Didactic Environment: Matching Pennies

Each player secretly chooses Heads (0) or Tails (1).
- If they **match**, Player 1 wins (+1 / -1).
- If they **differ**, Player 2 wins (-1 / +1).

The unique Nash equilibrium is the **uniform mixed strategy**: each player randomises 50/50.
Self-play should converge to this equilibrium.

In [ ]:
# ---------------------------------------------------------------
# Self-Play on Matching Pennies
# Both players use a softmax policy parameterised by a single logit.
# We track P(Heads) over training iterations.
# ---------------------------------------------------------------

class MatchingPenniesAgent:
    """Softmax policy with one free parameter (logit for Heads)."""

    def __init__(self, lr=0.05):
        self.logit = 0.0   # logit for action 0 (Heads); Tails logit fixed at 0
        self.lr = lr

    def prob_heads(self):
        return 1.0 / (1.0 + np.exp(-self.logit))   # sigmoid

    def sample(self):
        return 0 if random.random() < self.prob_heads() else 1

    def update(self, action, advantage):
        """REINFORCE update for a single-step episode."""
        p = self.prob_heads()
        # d/d_logit log pi(a|logit)
        if action == 0:
            grad = 1 - p   # d/d_logit log sigmoid(logit)
        else:
            grad = -p      # d/d_logit log (1 - sigmoid(logit))
        self.logit += self.lr * advantage * grad


def self_play_matching_pennies(n_iters=2000, update_freq=50, lr=0.1, seed=42):
    """
    Self-play loop.
    Every `update_freq` games, copy the main agent to become the new opponent.

    Returns
    -------
    p1_history : P(Heads) of the main agent over time
    win_rate   : win-rate of main agent vs frozen opponent (per block)
    """
    random.seed(seed)
    np.random.seed(seed)

    agent = MatchingPenniesAgent(lr=lr)
    opponent_logit = agent.logit   # frozen copy

    p1_history = []
    win_rate = []
    block_wins = 0

    for t in range(n_iters):
        # Opponent plays fixed policy
        p_opp = 1.0 / (1.0 + np.exp(-opponent_logit))
        opp_action = 0 if random.random() < p_opp else 1

        a1 = agent.sample()

        # Reward for player 1: +1 if match, -1 if differ
        r1 = +1.0 if a1 == opp_action else -1.0
        block_wins += (r1 > 0)

        agent.update(a1, r1)

        p1_history.append(agent.prob_heads())

        if (t + 1) % update_freq == 0:
            win_rate.append(block_wins / update_freq)
            block_wins = 0
            # Update opponent to current agent
            opponent_logit = agent.logit

    return p1_history, win_rate


p1_hist_sp, win_rate_sp = self_play_matching_pennies(n_iters=3000, update_freq=50, lr=0.08)
print(f'Final P(Heads): {p1_hist_sp[-1]:.4f}  (Nash equilibrium = 0.5000)')
print(f'Final win rate vs. own copy: {win_rate_sp[-1]:.3f}  (expected ≈ 0.500)')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

ax = axes[0]
ax.plot(smooth(p1_hist_sp, 50), color='#2E7D32', linewidth=2)
ax.axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Nash equilibrium (0.5)')
ax.set_xlabel('Training step')
ax.set_ylabel('P(Heads)')
ax.set_title('Self-Play: Agent Policy Convergence\n(Matching Pennies)')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
x_blocks = np.arange(len(win_rate_sp)) * 50
ax.plot(x_blocks, win_rate_sp, color='#E65100', linewidth=2)
ax.axhline(0.5, color='red', linestyle='--', linewidth=1.5, label='Expected (0.5)')
ax.set_xlabel('Training step')
ax.set_ylabel('Win rate vs. frozen opponent')
ax.set_title('Self-Play: Win Rate per Block')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print('\nKey insight: policy converges to the mixed Nash equilibrium (P=0.5) via self-play.')
print('Win rate converges to 0.5 — neither player can exploit the other.')

### 2.3 Two-Player Rock-Paper-Scissors with Self-Play

A richer illustration: RPS has a three-action Nash equilibrium at the uniform distribution (1/3, 1/3, 1/3).
We train a policy-gradient agent against itself and track the action distribution.

In [ ]:
# Payoff matrix for player 1 (Rock=0, Paper=1, Scissors=2)
# RPS_PAYOFF[a1][a2] = reward for player 1
RPS_PAYOFF = np.array([
    [ 0, -1,  1],   # Rock
    [ 1,  0, -1],   # Paper
    [-1,  1,  0],   # Scissors
])


class RPSAgent:
    """Softmax policy over 3 actions, trained with policy gradient."""

    def __init__(self, lr=0.05):
        self.logits = np.zeros(3)
        self.lr = lr

    def probs(self):
        e = np.exp(self.logits - self.logits.max())
        return e / e.sum()

    def sample(self):
        return np.random.choice(3, p=self.probs())

    def update(self, action, reward):
        p = self.probs()
        # Policy gradient: grad = (indicator - p) * reward
        grad = -p.copy()
        grad[action] += 1.0
        self.logits += self.lr * reward * grad


def rps_self_play(n_iters=5000, update_freq=100, lr=0.05):
    agent = RPSAgent(lr=lr)
    opponent_probs = agent.probs().copy()
    history = []

    for t in range(n_iters):
        opp_action = np.random.choice(3, p=opponent_probs)
        a = agent.sample()
        r = float(RPS_PAYOFF[a, opp_action])
        agent.update(a, r)
        history.append(agent.probs().copy())

        if (t + 1) % update_freq == 0:
            opponent_probs = agent.probs().copy()

    return np.array(history)


rps_history = rps_self_play(n_iters=5000, update_freq=100, lr=0.04)

fig, ax = plt.subplots(figsize=(11, 4))
labels = ['Rock', 'Paper', 'Scissors']
colors = ['#C62828', '#1565C0', '#2E7D32']
w = 30
for i, (lbl, col) in enumerate(zip(labels, colors)):
    ax.plot(smooth(rps_history[:, i], w), label=lbl, color=col, linewidth=2)
ax.axhline(1/3, color='black', linestyle='--', linewidth=1.5, label='Nash (1/3)')
ax.set_xlabel('Training step')
ax.set_ylabel('Action probability')
ax.set_title('RPS Self-Play — Action Probability Distribution')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_ylim(-0.05, 1.05)
plt.tight_layout()
plt.show()

print(f'Final distribution: Rock={rps_history[-1,0]:.3f}, '
      f'Paper={rps_history[-1,1]:.3f}, Scissors={rps_history[-1,2]:.3f}')
print('Nash equilibrium  : Rock=0.333, Paper=0.333, Scissors=0.333')

---
## Part 3 — Model-Based RL and Dyna-Q

### 3.1 Model-Free vs. Model-Based

| Property | Model-Free | Model-Based |
|----------|-----------|-------------|
| Learns | $V(s)$ or $\pi$ from experience | $\hat{P}(s'|s,a)$, $\hat{r}(s,a)$ |
| Sample efficiency | Low | High |
| Computation per step | Low | High |
| Bias | None from model | Model errors propagate |

### 3.2 World Models

A **world model** $\hat{\mathcal{M}}$ approximates the true environment:

$$\hat{s}', \hat{r} \leftarrow \hat{\mathcal{M}}(s, a)$$

Once learned, the agent can perform **mental simulations** (imagined rollouts) without interacting with the real environment, dramatically improving sample efficiency.

### 3.3 Dyna-Q

Sutton's **Dyna-Q** (1991) is the archetypal model-based algorithm:

```
For each real step:
  1. Take action a, observe r, s'
  2. Q-learning update on (s, a, r, s')          ← direct RL
  3. Update model: model[s][a] = (r, s')          ← model learning
  4. Repeat n times:
       Sample random (s, a) from model
       Q-learning update using model(s, a)        ← planning
```

The $n$ planning steps provide $n$ extra Q-updates per real step — boosting sample efficiency by a factor of approximately $n+1$.

In [ ]:
# ---------------------------------------------------------------
# Dyna-Q vs plain Q-Learning on FrozenLake-v1
# We compare the number of episodes needed to reach a target
# cumulative reward, varying the number of planning steps n.
# ---------------------------------------------------------------

def run_dyna_q(env_name='FrozenLake-v1', n_planning=5, n_episodes=500,
               alpha=0.1, gamma=0.99, eps=0.1, seed=0):
    """
    Dyna-Q on a tabular environment.

    Parameters
    ----------
    n_planning : int
        Number of simulated (planning) Q-updates per real step.
        n_planning=0 reduces to plain Q-learning.

    Returns
    -------
    rewards : list of cumulative episode rewards
    """
    env = gym.make(env_name, is_slippery=False)  # deterministic for clarity
    n_s = env.observation_space.n
    n_a = env.action_space.n

    Q = np.zeros((n_s, n_a))
    # Model: model[s][a] = (reward, next_state) — only populated after visits
    model = {}   # dict of {s: {a: (r, s')}}
    visited = []  # list of (s, a) pairs seen so far

    episode_rewards = []
    rng = np.random.default_rng(seed)

    for ep in range(n_episodes):
        state, _ = env.reset(seed=seed + ep)
        total_r = 0.0
        done = False

        while not done:
            # Epsilon-greedy
            if rng.random() < eps:
                action = env.action_space.sample()
            else:
                action = int(np.argmax(Q[state]))

            next_state, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_r += reward

            # --- Direct RL update ---
            best_next = np.max(Q[next_state]) if not done else 0.0
            td_err = reward + gamma * best_next - Q[state, action]
            Q[state, action] += alpha * td_err

            # --- Model update ---
            if state not in model:
                model[state] = {}
            if action not in model[state]:
                visited.append((state, action))
            model[state][action] = (reward, next_state, done)

            # --- Planning steps ---
            for _ in range(n_planning):
                if not visited:
                    break
                idx = rng.integers(len(visited))
                ps, pa = visited[idx]
                pr, pnext, pdone = model[ps][pa]
                pbest_next = np.max(Q[pnext]) if not pdone else 0.0
                ptd = pr + gamma * pbest_next - Q[ps, pa]
                Q[ps, pa] += alpha * ptd

            state = next_state

        episode_rewards.append(total_r)

    env.close()
    return episode_rewards


print('Running Dyna-Q experiments (this may take ~20 s)...')
configs = [
    (0,  '#B71C1C', 'Q-Learning (n=0)'),
    (5,  '#F57F17', 'Dyna-Q  (n=5)'),
    (20, '#1B5E20', 'Dyna-Q  (n=20)'),
    (50, '#1A237E', 'Dyna-Q  (n=50)'),
]

results_dyna = {}
for n, color, label in configs:
    # Average over 5 seeds for reliability
    runs = [run_dyna_q(n_planning=n, n_episodes=300, seed=s) for s in range(5)]
    results_dyna[(n, label, color)] = np.mean(runs, axis=0)
    print(f'  n={n:2d} done — final 50-ep success rate: '
          f'{np.mean(results_dyna[(n, label, color)][-50:]):.3f}')

In [ ]:
w = 20
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for (n, label, color), rewards in results_dyna.items():
    ax.plot(smooth(rewards, w), label=label, color=color, linewidth=2)
ax.set_xlabel('Episode')
ax.set_ylabel('Success rate (smoothed)')
ax.set_title('Dyna-Q: Effect of Planning Steps\n(FrozenLake, averaged over 5 seeds)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# Cumulative success — real-environment steps to first convergence
ax = axes[1]
n_vals = [k[0] for k in results_dyna]
cum_success = []
for (n, label, color), rewards in results_dyna.items():
    cum_success.append(np.cumsum(rewards)[-1])

bars = ax.bar([k[2] for k in results_dyna], cum_success,
              color=[k[1] for k in results_dyna], width=0.5, alpha=0.85)
ax.set_ylabel('Total successes in 300 episodes')
ax.set_title('Total Successes vs. Planning Budget')
ax.tick_params(axis='x', rotation=20)
for bar, val in zip(bars, cum_success):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{int(val)}', ha='center', va='bottom', fontsize=11)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nKey insight: each additional planning step is "free" in terms of environment '
      'interactions.\nDyna-Q with n=50 reaches peak performance much faster than '
      'plain Q-learning.')

---
## Part 4 — Offline RL & Imitation Learning

### 4.1 The Offline RL Setting

In **offline RL** (also called batch RL), the agent has access to a fixed dataset $\mathcal{D} = \{(s_t, a_t, r_t, s_{t+1})\}$ collected by some **behaviour policy** $\mu$ but **cannot interact** with the environment.

$$\text{Offline RL:} \quad \pi^* = \arg\max_\pi \mathbb{E}_{\tau \sim \pi}\left[\sum_t \gamma^t r_t\right] \quad \text{given } \mathcal{D} \sim \mu$$

This is critical for:
- **Healthcare** — we cannot run random policies on patients.
- **Autonomous driving** — safety constraints prevent exploration.
- **Industrial control** — costly real-world interactions.

### 4.2 Imitation Learning via Behavioral Cloning

**Behavioral Cloning (BC)** is the simplest offline approach: treat the dataset as a supervised learning problem and fit a policy by minimising

$$\mathcal{L}_{\text{BC}}(\theta) = -\mathbb{E}_{(s,a) \sim \mathcal{D}}\left[\log \pi_\theta(a | s)\right]$$

BC is easy to implement but suffers from **distribution shift**: at test time, small errors compound and the agent visits states not in $\mathcal{D}$.

In [ ]:
# ---------------------------------------------------------------
# Behavioral Cloning on CartPole-v1
#
# Step 1: Generate expert demonstrations using a hand-crafted policy.
# Step 2: Train a neural network clone on those demonstrations.
# Step 3: Evaluate the clone and compare to the expert.
# ---------------------------------------------------------------

# -------- Expert policy (simple heuristic) --------
def expert_action(obs):
    """
    Heuristic expert for CartPole:
    Push cart in the direction the pole is falling.
    obs = [cart_pos, cart_vel, pole_angle, pole_ang_vel]
    """
    pole_angle = obs[2]
    pole_ang_vel = obs[3]
    # Weighted combination of angle and angular velocity
    return 1 if (pole_angle + 0.5 * pole_ang_vel) > 0 else 0


def collect_demonstrations(n_episodes=50, seed=0):
    """Collect expert trajectories and return (states, actions) arrays."""
    env = gym.make('CartPole-v1')
    states, actions = [], []
    expert_returns = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        ep_return = 0.0
        done = False
        while not done:
            a = expert_action(obs)
            states.append(obs)
            actions.append(a)
            obs, r, term, trunc, _ = env.step(a)
            done = term or trunc
            ep_return += r
        expert_returns.append(ep_return)

    env.close()
    print(f'Expert — {n_episodes} episodes | '
          f'mean return: {np.mean(expert_returns):.1f} ± {np.std(expert_returns):.1f}')
    return np.array(states, dtype=np.float32), np.array(actions, dtype=np.int64)


demo_states, demo_actions = collect_demonstrations(n_episodes=80, seed=SEED)
print(f'Dataset size: {len(demo_states)} transitions')

In [ ]:
# -------- Behavioral Cloning network --------

class BCPolicy(nn.Module):
    """MLP policy trained by supervised imitation."""

    def __init__(self, state_dim=4, n_actions=2, hidden=64):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )

    def forward(self, x):
        return self.net(x)

    def act(self, obs):
        with torch.no_grad():
            obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
            logits = self.forward(obs_t)
            return int(logits.argmax(dim=-1).item())


def train_bc(states, actions, n_epochs=30, batch_size=64, lr=1e-3):
    """Train behavioral cloning policy via cross-entropy."""
    policy = BCPolicy().to(device)
    optimizer = optim.Adam(policy.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    S = torch.FloatTensor(states).to(device)
    A = torch.LongTensor(actions).to(device)
    n = len(S)

    loss_history = []
    for epoch in range(n_epochs):
        # Shuffle
        idx = torch.randperm(n)
        S, A = S[idx], A[idx]
        epoch_loss = 0.0

        for start in range(0, n, batch_size):
            sb = S[start:start + batch_size]
            ab = A[start:start + batch_size]
            logits = policy(sb)
            loss = criterion(logits, ab)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(sb)

        avg_loss = epoch_loss / n
        loss_history.append(avg_loss)
        if (epoch + 1) % 10 == 0:
            print(f'  Epoch {epoch+1:3d} | Loss: {avg_loss:.4f}')

    return policy, loss_history


print('Training Behavioral Cloning policy...')
bc_policy, bc_loss = train_bc(demo_states, demo_actions, n_epochs=50)


def evaluate_policy_fn(policy_fn, env_name='CartPole-v1', n_episodes=20, seed=100):
    """Evaluate a callable policy and return mean, std episode return."""
    env = gym.make(env_name)
    returns = []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total_r = 0.0
        done = False
        while not done:
            a = policy_fn(obs)
            obs, r, term, trunc, _ = env.step(a)
            done = term or trunc
            total_r += r
        returns.append(total_r)
    env.close()
    return np.mean(returns), np.std(returns)


bc_mean, bc_std = evaluate_policy_fn(bc_policy.act)
expert_mean, expert_std = evaluate_policy_fn(expert_action)

print(f'\nExpert     — mean return: {expert_mean:.1f} ± {expert_std:.1f}')
print(f'BC clone   — mean return: {bc_mean:.1f} ± {bc_std:.1f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Training loss
ax = axes[0]
ax.plot(bc_loss, color='#6A1B9A', linewidth=2)
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.set_title('Behavioral Cloning — Training Loss')
ax.grid(True, alpha=0.3)

# Performance bar chart
ax = axes[1]
labels_ = ['Expert Policy', 'BC Clone']
means_ = [expert_mean, bc_mean]
stds_  = [expert_std,  bc_std]
colors_ = ['#1B5E20', '#6A1B9A']
bars_ = ax.bar(labels_, means_, yerr=stds_, color=colors_, alpha=0.8,
               capsize=8, width=0.4)
ax.set_ylabel('Mean Episode Return')
ax.set_title('Evaluation: Expert vs. Behavioral Clone')
ax.set_ylim(0, 520)
ax.axhline(500, color='gray', linestyle='--', alpha=0.5, label='Max possible')
for bar, val in zip(bars_, means_):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            f'{val:.0f}', ha='center', fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print('\nBC succeeds here because CartPole has a simple dynamics and the '
      'expert covers most of the state space.\n'
      'On harder tasks, distribution shift degrades performance — '
      'motivating DAgger, IRL, and offline RL methods like CQL and TD3+BC.')

---
## Part 5 — Capstone Project: RL for African Energy Grid Resource Allocation

### 5.1 Problem Motivation

Sub-Saharan Africa faces a persistent energy access challenge. Many grids are composed of **microgrids** powered by solar panels, battery storage, and diesel generators. Each hour, an operator must decide:

- How much energy to **store** in the battery vs. **dispatch** to consumers.
- When to run the **diesel generator** (expensive, polluting — last resort).
- How to handle **demand peaks** and variable **solar generation**.

This is a classic **sequential decision problem under uncertainty** — exactly the kind RL excels at.

### 5.2 Custom Gymnasium Environment: `AfricanMicrogridEnv`

**State** $s_t$:
| Variable | Description | Range |
|----------|-------------|-------|
| `battery_soc` | State of Charge (fraction of capacity) | [0, 1] |
| `solar_gen` | Solar generation (normalised kW) | [0, 1] |
| `demand` | Consumer demand (normalised kW) | [0, 1] |
| `hour_of_day` | Hour encoded as sine/cosine | 2 values |

**Action** $a_t$ (discrete):
| Index | Meaning |
|-------|---------|
| 0 | Charge battery from solar |
| 1 | Discharge battery to meet demand |
| 2 | Use solar directly + do nothing |
| 3 | Start diesel generator |

**Reward** $r_t$: penalises unmet demand (blackout), diesel use, and battery damage; rewards efficient solar utilisation.

In [ ]:
class AfricanMicrogridEnv(gym.Env):
    """
    Simplified African microgrid resource allocation environment.

    The agent controls energy dispatch over a 24-hour horizon.
    Each episode = 1 simulated day (24 steps, each step = 1 hour).

    Observation (5-dim, all floats in [0,1] except sin/cos):
        [battery_soc, solar_gen, demand, sin(2π·h/24), cos(2π·h/24)]

    Actions (discrete, 4):
        0 = Charge battery   (solar → battery)
        1 = Discharge battery (battery → grid)
        2 = Direct solar     (solar → grid, battery idle)
        3 = Diesel generator (diesel → grid, emergency)

    Reward components:
        +1.5  per unit of demand met from solar (green reward)
        +1.0  per unit of demand met from battery
        -0.5  per unit of demand met from diesel (cost + carbon)
        -3.0  per unit of unmet demand (blackout penalty)
        -0.1  battery degradation when SoC > 0.95 or < 0.05
    """

    metadata = {'render_modes': []}

    # Grid parameters
    BATTERY_CAP  = 1.0   # normalised capacity
    CHARGE_RATE  = 0.2   # fraction of capacity charged per step
    DIESEL_CAP   = 1.0   # diesel can always cover full demand

    def __init__(self):
        super().__init__()
        self.observation_space = spaces.Box(
            low=np.array([0.0, 0.0, 0.0, -1.0, -1.0], dtype=np.float32),
            high=np.array([1.0, 1.0, 1.0,  1.0,  1.0], dtype=np.float32)
        )
        self.action_space = spaces.Discrete(4)
        self.rng = np.random.default_rng()

    # ------------------------------------------------------------------
    # Internal helpers
    # ------------------------------------------------------------------

    def _solar_profile(self, hour):
        """Idealised tropical solar curve peaked at noon."""
        if 6 <= hour <= 18:
            base = np.sin(np.pi * (hour - 6) / 12)
        else:
            base = 0.0
        # Add stochastic cloud cover
        noise = self.rng.normal(0, 0.05)
        return float(np.clip(base + noise, 0.0, 1.0))

    def _demand_profile(self, hour):
        """Demand peaks in the morning and evening."""
        # Morning peak 7–9 h, evening peak 18–21 h
        morning = 0.6 * np.exp(-0.5 * ((hour - 8) / 1.5) ** 2)
        evening = 0.9 * np.exp(-0.5 * ((hour - 19) / 2.0) ** 2)
        base = 0.2
        noise = self.rng.normal(0, 0.05)
        return float(np.clip(base + morning + evening + noise, 0.0, 1.0))

    def _get_obs(self):
        h = self.hour
        return np.array([
            self.battery_soc,
            self._solar_profile(h),
            self._demand_profile(h),
            np.sin(2 * np.pi * h / 24),
            np.cos(2 * np.pi * h / 24)
        ], dtype=np.float32)

    # ------------------------------------------------------------------
    # Gymnasium API
    # ------------------------------------------------------------------

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.rng = np.random.default_rng(seed)
        self.hour = 0
        self.battery_soc = self.rng.uniform(0.3, 0.7)  # random initial SoC
        self.total_reward = 0.0
        self.history = []
        obs = self._get_obs()
        return obs, {}

    def step(self, action):
        solar  = self._solar_profile(self.hour)
        demand = self._demand_profile(self.hour)
        reward = 0.0
        info   = {}

        if action == 0:   # Charge battery from solar
            charge_amount = min(solar, self.CHARGE_RATE,
                                self.BATTERY_CAP - self.battery_soc)
            self.battery_soc += charge_amount
            # Remaining solar to direct grid
            solar_to_grid = solar - charge_amount
            met = min(solar_to_grid, demand)
            unmet = max(demand - met, 0.0)
            reward += 1.5 * met - 3.0 * unmet

        elif action == 1:  # Discharge battery
            discharge = min(self.battery_soc, demand)
            self.battery_soc -= discharge
            remaining = demand - discharge
            # Use solar for remainder
            solar_met = min(solar, remaining)
            unmet = max(remaining - solar_met, 0.0)
            reward += 1.0 * discharge + 1.5 * solar_met - 3.0 * unmet

        elif action == 2:  # Direct solar
            met = min(solar, demand)
            unmet = max(demand - met, 0.0)
            reward += 1.5 * met - 3.0 * unmet

        elif action == 3:  # Diesel generator
            reward += -0.5 * demand   # cost of diesel fuel per unit demand
            # Diesel always meets full demand
            reward += 0.0             # no blackout penalty
            # Any surplus solar can still charge battery
            surplus = max(solar - demand, 0.0)
            charge = min(surplus, self.CHARGE_RATE,
                         self.BATTERY_CAP - self.battery_soc)
            self.battery_soc += charge

        # Battery degradation at extremes
        if self.battery_soc > 0.95 or self.battery_soc < 0.05:
            reward -= 0.1

        self.battery_soc = float(np.clip(self.battery_soc, 0.0, 1.0))

        # Record history
        self.history.append({
            'hour': self.hour, 'solar': solar, 'demand': demand,
            'action': action, 'battery_soc': self.battery_soc, 'reward': reward
        })

        self.hour += 1
        terminated = (self.hour >= 24)
        obs = self._get_obs() if not terminated else np.zeros(5, dtype=np.float32)
        self.total_reward += reward
        return obs, reward, terminated, False, info


# -------- Sanity check --------
env_mg = AfricanMicrogridEnv()
obs, _ = env_mg.reset(seed=0)
print(f'Observation space : {env_mg.observation_space}')
print(f'Action space      : {env_mg.action_space}')
print(f'Initial obs       : {obs}')

# Quick random rollout
total_r = 0.0
done = False
while not done:
    a = env_mg.action_space.sample()
    obs, r, done, _, _ = env_mg.step(a)
    total_r += r
print(f'Random policy return (1 day): {total_r:.3f}')

In [ ]:
# ---------------------------------------------------------------
# Visualise 24-hour profiles (solar, demand, action choices)
# ---------------------------------------------------------------
env_mg2 = AfricanMicrogridEnv()
env_mg2.reset(seed=7)
done = False
while not done:
    _, _, done, _, _ = env_mg2.step(env_mg2.action_space.sample())
hist = env_mg2.history

hours   = [h['hour']        for h in hist]
solar   = [h['solar']       for h in hist]
demand  = [h['demand']      for h in hist]
soc     = [h['battery_soc'] for h in hist]
actions = [h['action']      for h in hist]
rewards = [h['reward']      for h in hist]

action_labels = ['Charge', 'Discharge', 'Direct Solar', 'Diesel']
action_colors = ['#1565C0', '#F57F17', '#2E7D32', '#B71C1C']

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

ax = axes[0]
ax.fill_between(hours, solar, alpha=0.4, color='#F9A825', label='Solar generation')
ax.fill_between(hours, demand, alpha=0.4, color='#1565C0', label='Consumer demand')
ax.set_ylabel('Power (normalised)')
ax.set_title('African Microgrid — 24-Hour Simulation (Random Policy)')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(hours, soc, color='#6A1B9A', linewidth=2.5, label='Battery SoC')
ax.axhline(0.95, color='red', linestyle=':', alpha=0.7, label='Degradation zone')
ax.axhline(0.05, color='red', linestyle=':', alpha=0.7)
ax.scatter(hours, soc, c=[action_colors[a] for a in actions], zorder=5, s=60)
ax.set_ylabel('State of Charge')
ax.set_ylim(-0.05, 1.1)
ax.legend(loc='upper left')
patches = [mpatches.Patch(color=action_colors[i], label=action_labels[i]) for i in range(4)]
ax.legend(handles=patches, loc='upper right', fontsize=9)
ax.grid(True, alpha=0.3)

ax = axes[2]
cumr = np.cumsum(rewards)
ax.bar(hours, rewards, color=['#2E7D32' if r >= 0 else '#B71C1C' for r in rewards],
       alpha=0.7, label='Step reward')
ax.plot(hours, cumr, color='black', linewidth=2, label='Cumulative reward')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Reward')
ax.legend(loc='upper left')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# Train a DQN agent on AfricanMicrogridEnv
# We implement a lightweight DQN from scratch.
# ---------------------------------------------------------------

class QNetwork(nn.Module):
    """Q-network: maps state to Q(s, a) for all a."""

    def __init__(self, state_dim=5, n_actions=4, hidden=128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden),    nn.ReLU(),
            nn.Linear(hidden, n_actions)
        )

    def forward(self, x):
        return self.net(x)


class ReplayBuffer:
    """Experience replay buffer."""

    def __init__(self, capacity=10000):
        self.buf = deque(maxlen=capacity)

    def push(self, s, a, r, s_next, done):
        self.buf.append((s, a, r, s_next, done))

    def sample(self, batch_size):
        batch = random.sample(self.buf, batch_size)
        s, a, r, s2, d = zip(*batch)
        return (torch.FloatTensor(np.array(s)).to(device),
                torch.LongTensor(a).to(device),
                torch.FloatTensor(r).to(device),
                torch.FloatTensor(np.array(s2)).to(device),
                torch.FloatTensor(d).to(device))

    def __len__(self):
        return len(self.buf)


def train_microgrid_dqn(n_episodes=600, gamma=0.99, lr=3e-4,
                        batch_size=128, eps_start=1.0, eps_end=0.05,
                        eps_decay=0.995, target_update=10, seed=SEED):
    """Train DQN on AfricanMicrogridEnv."""
    env = AfricanMicrogridEnv()
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    q_net    = QNetwork().to(device)
    q_target = QNetwork().to(device)
    q_target.load_state_dict(q_net.state_dict())
    q_target.eval()

    optimizer = optim.Adam(q_net.parameters(), lr=lr)
    buffer = ReplayBuffer(capacity=20000)

    eps = eps_start
    ep_rewards = []
    ep_solar_used = []
    ep_diesel_used = []

    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total_r = 0.0
        done = False

        while not done:
            if random.random() < eps:
                action = env.action_space.sample()
            else:
                with torch.no_grad():
                    obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
                    action = int(q_net(obs_t).argmax(dim=1).item())

            next_obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            buffer.push(obs, action, reward, next_obs, float(done))
            obs = next_obs
            total_r += reward

            # Train if enough samples
            if len(buffer) >= batch_size:
                s_b, a_b, r_b, s2_b, d_b = buffer.sample(batch_size)
                with torch.no_grad():
                    next_q = q_target(s2_b).max(dim=1).values
                    targets = r_b + gamma * next_q * (1 - d_b)
                current_q = q_net(s_b).gather(1, a_b.unsqueeze(1)).squeeze()
                loss = F.smooth_l1_loss(current_q, targets)
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(q_net.parameters(), 1.0)
                optimizer.step()

        eps = max(eps_end, eps * eps_decay)
        ep_rewards.append(total_r)

        # Track action usage over last episode
        actions_ep = [h['action'] for h in env.history]
        ep_solar_used.append(actions_ep.count(2) + actions_ep.count(0))
        ep_diesel_used.append(actions_ep.count(3))

        if (ep + 1) % target_update == 0:
            q_target.load_state_dict(q_net.state_dict())

        if (ep + 1) % 100 == 0:
            avg = np.mean(ep_rewards[-100:])
            diesel_avg = np.mean(ep_diesel_used[-100:])
            print(f'Episode {ep+1:4d} | Avg reward: {avg:6.2f} | '
                  f'Diesel uses/day: {diesel_avg:.1f} | eps: {eps:.3f}')

    return ep_rewards, ep_solar_used, ep_diesel_used, q_net


print('Training DQN on AfricanMicrogridEnv...')
mg_rewards, mg_solar, mg_diesel, dqn_agent = train_microgrid_dqn(n_episodes=600)

In [ ]:
# ---------------------------------------------------------------
# Capstone results: compare random policy vs. trained DQN
# ---------------------------------------------------------------

def run_policy_mg(policy_fn, n_episodes=50, seed=200):
    """Run a policy on the microgrid env and collect returns + stats."""
    env = AfricanMicrogridEnv()
    returns, diesel_counts, solar_counts = [], [], []
    for ep in range(n_episodes):
        obs, _ = env.reset(seed=seed + ep)
        total_r = 0.0
        done = False
        while not done:
            a = policy_fn(obs)
            obs, r, done, _, _ = env.step(a)
            total_r += r
        returns.append(total_r)
        actions_ep = [h['action'] for h in env.history]
        diesel_counts.append(actions_ep.count(3))
        solar_counts.append(actions_ep.count(2) + actions_ep.count(0))
    return returns, diesel_counts, solar_counts


def dqn_policy(obs):
    dqn_agent.eval()
    with torch.no_grad():
        obs_t = torch.FloatTensor(obs).unsqueeze(0).to(device)
        return int(dqn_agent(obs_t).argmax(dim=1).item())

def random_policy(obs):
    return np.random.randint(4)

# Greedy heuristic: use solar during day, battery at night, diesel never
def heuristic_policy(obs):
    soc, solar, demand, sin_h, cos_h = obs
    hour = (np.arctan2(sin_h, cos_h) / (2 * np.pi) * 24) % 24
    if solar > 0.15 and demand < solar:
        return 0   # charge battery when solar surplus
    elif solar > 0.15:
        return 2   # direct solar
    elif soc > 0.15:
        return 1   # discharge battery
    else:
        return 3   # fallback diesel


print('Evaluating policies...')
rand_ret, rand_diesel, rand_solar = run_policy_mg(random_policy)
heur_ret, heur_diesel, heur_solar = run_policy_mg(heuristic_policy)
dqn_ret,  dqn_diesel,  dqn_solar  = run_policy_mg(dqn_policy)

for name, ret, diesel, solar_c in [
    ('Random',    rand_ret, rand_diesel, rand_solar),
    ('Heuristic', heur_ret, heur_diesel, heur_solar),
    ('DQN',       dqn_ret,  dqn_diesel,  dqn_solar)
]:
    print(f'{name:12s} | Return: {np.mean(ret):6.2f} ± {np.std(ret):.2f} '
          f'| Diesel hrs/day: {np.mean(diesel):.1f} '
          f'| Solar hrs/day: {np.mean(solar_c):.1f}')

In [ ]:
w = 30
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Training curve
ax = axes[0]
ax.plot(smooth(mg_rewards, w), color='#1565C0', linewidth=2, label='DQN')
ax.set_xlabel('Episode')
ax.set_ylabel('Daily Return')
ax.set_title('DQN Training — Microgrid')
ax.grid(True, alpha=0.3)

# Diesel usage over training
ax = axes[1]
ax.plot(smooth(mg_diesel, w), color='#B71C1C', linewidth=2, label='Diesel hours/day')
ax.set_xlabel('Episode')
ax.set_ylabel('Diesel activations per day')
ax.set_title('Diesel Usage During Training\n(decreases = greener policy)')
ax.grid(True, alpha=0.3)

# Policy comparison bar chart
ax = axes[2]
policies = ['Random', 'Heuristic', 'DQN']
means_  = [np.mean(rand_ret), np.mean(heur_ret), np.mean(dqn_ret)]
stds_   = [np.std(rand_ret),  np.std(heur_ret),  np.std(dqn_ret)]
colors_ = ['#90A4AE', '#F57F17', '#1B5E20']
bars    = ax.bar(policies, means_, yerr=stds_, color=colors_,
                 alpha=0.85, capsize=8, width=0.45)
for bar, val in zip(bars, means_):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val:.1f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylabel('Mean Daily Return')
ax.set_title('Policy Comparison — Microgrid')
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Capstone: African Microgrid Energy Management', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print('\nKey result: DQN learns to prefer solar and battery dispatch,\n'
      'reducing diesel usage and avoiding blackout penalties.')

In [ ]:
# ---------------------------------------------------------------
# Visualise a trained DQN episode — 24-hour dispatch schedule
# ---------------------------------------------------------------
env_vis2 = AfricanMicrogridEnv()
obs, _ = env_vis2.reset(seed=300)
done = False
while not done:
    a = dqn_policy(obs)
    obs, _, done, _, _ = env_vis2.step(a)
hist_dqn = env_vis2.history

hours_d   = [h['hour']        for h in hist_dqn]
solar_d   = [h['solar']       for h in hist_dqn]
demand_d  = [h['demand']      for h in hist_dqn]
soc_d     = [h['battery_soc'] for h in hist_dqn]
actions_d = [h['action']      for h in hist_dqn]
rewards_d = [h['reward']      for h in hist_dqn]

fig, axes = plt.subplots(3, 1, figsize=(13, 10), sharex=True)

ax = axes[0]
ax.fill_between(hours_d, solar_d, alpha=0.4, color='#F9A825', label='Solar')
ax.fill_between(hours_d, demand_d, alpha=0.35, color='#1565C0', label='Demand')
ax.set_ylabel('Power (normalised)')
ax.set_title('DQN Agent — Optimal 24-Hour Dispatch Schedule')
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
ax.plot(hours_d, soc_d, color='#6A1B9A', linewidth=2.5)
ax.axhline(0.95, color='red', linestyle=':', alpha=0.5)
ax.axhline(0.05, color='red', linestyle=':', alpha=0.5)
scatter_colors = [action_colors[a] for a in actions_d]
ax.scatter(hours_d, soc_d, c=scatter_colors, zorder=5, s=80)
patches = [mpatches.Patch(color=action_colors[i], label=action_labels[i]) for i in range(4)]
ax.legend(handles=patches, loc='upper right', fontsize=9)
ax.set_ylabel('Battery SoC')
ax.set_ylim(-0.05, 1.1)
ax.grid(True, alpha=0.3)

ax = axes[2]
ax.bar(hours_d, rewards_d,
       color=['#2E7D32' if r >= 0 else '#B71C1C' for r in rewards_d],
       alpha=0.75)
ax.plot(hours_d, np.cumsum(rewards_d), color='black', linewidth=2, label='Cumulative')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Reward')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'DQN daily return: {sum(rewards_d):.3f}')
print(f'Diesel activations: {actions_d.count(3)}')
print(f'Solar activations:  {actions_d.count(2) + actions_d.count(0)}')

---
## Part 6 — Workshop Summary & Research Frontiers

### 6.1 Five-Day Recap

| Day | Theme | Key Algorithms |
|-----|-------|----------------|
| **1** | MDPs & Dynamic Programming | Policy Evaluation, Policy Iteration, Value Iteration |
| **2** | Tabular Methods | Q-Learning, SARSA, Monte Carlo, Eligibility Traces |
| **3** | Deep Q-Networks | DQN, Double-DQN, Dueling DQN, Prioritised Experience Replay |
| **4** | Policy Gradients | REINFORCE, A2C, PPO |
| **5** | Advanced Topics | MARL, Self-Play, Dyna-Q, Behavioral Cloning, Custom Environments |

### 6.2 The RL Algorithm Landscape

```
                    ┌─────────────────────────────────────┐
                    │       Reinforcement Learning         │
                    └───────────────┬─────────────────────┘
                                    │
                   ┌────────────────┴───────────────┐
                   │                                │
            Model-Free                         Model-Based
           ┌───────┴──────┐                  (Dyna, MBPO, Dreamer)
           │              │
      Value-Based    Policy-Based
      (Q-learning,   ┌────┴────┐
       DQN, C51)     │         │
                  On-Policy  Off-Policy
                  (REINFORCE, (SAC, TD3,
                   A2C, PPO)   DDPG)
```

### 6.3 Research Frontiers (2024–2025)

#### Foundation Models meet RL
- **Decision Transformer** — offline RL as sequence modelling with transformers.
- **GATO (DeepMind)** — one model for 600+ tasks across RL, NLP, and vision.
- **Large Language Models as planners** (SayCan, ReAct, Voyager).

#### Safe and Constrained RL
- **Constrained MDPs (CMDPs)** — optimise reward subject to safety constraints.
- **RLHF** — RL from human feedback (how ChatGPT was aligned).

#### Multi-Agent RL
- **MAPPO** — centralised training with decentralised execution.
- **QMIX, QTRAN** — cooperative MARL with monotone value factorisation.
- **AlphaStar, OpenAI Five** — superhuman performance in complex games.

#### Applications in Africa & Global South
- **Energy access** — microgrid management (our capstone), demand response.
- **Agriculture** — crop irrigation scheduling under weather uncertainty.
- **Healthcare logistics** — vaccine cold-chain optimisation.
- **Traffic management** — adaptive signal control in rapidly-growing cities.

### 6.4 Recommended Papers

| Paper | Year | Contribution |
|-------|------|-------------|
| Mnih et al., *Human-level control through DRL* | 2015 | DQN |
| Schulman et al., *Proximal Policy Optimization* | 2017 | PPO |
| Silver et al., *Mastering the game of Go without human knowledge* | 2017 | AlphaGo Zero / Self-Play |
| Haarnoja et al., *Soft Actor-Critic* | 2018 | SAC (max-entropy RL) |
| Lowe et al., *Multi-Agent Actor-Critic for Mixed Cooperative-Competitive* | 2017 | MADDPG |
| Chen et al., *Decision Transformer* | 2021 | Offline RL as seq. modelling |
| Reed et al., *A Generalist Agent (GATO)* | 2022 | Multi-task foundation model |
| Ouyang et al., *Training language models to follow instructions (InstructGPT)* | 2022 | RLHF |

### 6.5 Recommended Textbooks

1. **Sutton & Barto** — *Reinforcement Learning: An Introduction* (2nd ed., free online)
2. **Bertsekas** — *Reinforcement Learning and Optimal Control*
3. **Szepesvári** — *Algorithms for Reinforcement Learning* (free online)

### 6.6 Code Ecosystem

| Tool | Purpose |
|------|---------|
| `gymnasium` | Standard RL environment API |
| `stable-baselines3` | Production-ready algorithm implementations |
| `RLlib` (Ray) | Scalable distributed RL |
| `CleanRL` | Single-file, readable reference implementations |
| `PettingZoo` | Multi-agent environment standard |

In [ ]:
# ---------------------------------------------------------------
# Final visualisation: algorithm family tree with performance context
# ---------------------------------------------------------------

fig, ax = plt.subplots(figsize=(13, 7))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis('off')
ax.set_facecolor('#FAFAFA')
fig.patch.set_facecolor('#FAFAFA')

# Title
ax.text(5, 9.5, 'RL Algorithm Landscape — Workshop Summary',
        ha='center', va='center', fontsize=14, fontweight='bold')

def box(ax, x, y, text, color, fontsize=10, width=1.9, height=0.55):
    rect = mpatches.FancyBboxPatch(
        (x - width/2, y - height/2), width, height,
        boxstyle='round,pad=0.1', facecolor=color, edgecolor='white',
        linewidth=2, alpha=0.92
    )
    ax.add_patch(rect)
    ax.text(x, y, text, ha='center', va='center',
            fontsize=fontsize, color='white', fontweight='bold')

def arrow(ax, x1, y1, x2, y2):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='#455A64', lw=1.5))

# Root
box(ax, 5, 8.6, 'Reinforcement Learning', '#1A237E', fontsize=11, width=3.0)

# Level 1
box(ax, 2.5, 7.4, 'Model-Free', '#283593', width=2.2)
box(ax, 7.5, 7.4, 'Model-Based', '#4527A0', width=2.2)
arrow(ax, 5, 8.3, 2.5, 7.7)
arrow(ax, 5, 8.3, 7.5, 7.7)

# Level 2
box(ax, 1.0, 6.0, 'Value-Based', '#0277BD', width=1.8)
box(ax, 4.0, 6.0, 'Policy-Based', '#00695C', width=1.8)
box(ax, 7.5, 6.0, 'World Models\n(Dyna-Q, MBPO)', '#4527A0', width=2.2, height=0.7)
arrow(ax, 2.5, 7.1, 1.0, 6.3)
arrow(ax, 2.5, 7.1, 4.0, 6.3)
arrow(ax, 7.5, 7.1, 7.5, 6.35)

# Level 3 — algorithms
# Value-based
for i, (name, col) in enumerate([
    ('Q-Learning\n(Day 2)', '#01579B'),
    ('DQN\n(Day 3)', '#0288D1'),
    ('Double DQN\nDueling DQN', '#29B6F6'),
]):
    bx = 0.9 + i * 0.0
    by = 4.5 - i * 1.2
    box(ax, 1.0, by, name, col, fontsize=8.5, width=1.65, height=0.65)
    arrow(ax, 1.0, 5.7 - i * 1.2, 1.0, 5.0 - i * 1.2)

# Policy-based
for i, (name, col) in enumerate([
    ('REINFORCE\n(Day 4)', '#1B5E20'),
    ('A2C\n(Day 4)', '#2E7D32'),
    ('PPO\n(Day 4)', '#43A047'),
]):
    by = 4.5 - i * 1.2
    box(ax, 4.0, by, name, col, fontsize=8.5, width=1.65, height=0.65)
    arrow(ax, 4.0, 5.7 - i * 1.2, 4.0, 5.0 - i * 1.2)

# Multi-agent
box(ax, 7.5, 4.5, 'Multi-Agent RL\n(Day 5)', '#6A1B9A', width=2.2, height=0.7)
arrow(ax, 7.5, 5.65, 7.5, 4.85)
for i, (name, col) in enumerate([
    ('IQL / Self-Play', '#7B1FA2'),
    ('MAPPO / QMIX', '#9C27B0'),
]):
    by = 3.4 - i * 1.0
    box(ax, 7.5, by, name, col, fontsize=8.5, width=2.2, height=0.6)
    arrow(ax, 7.5, 4.15 - i * 1.0, 7.5, 3.7 - i * 1.0)

# Offline / IL
box(ax, 5.0, 2.2, 'Offline RL & Imitation Learning (Day 5)', '#BF360C',
    fontsize=9, width=4.5, height=0.6)

# Day labels
for xp, yp, txt in [
    (1.0, 0.8, 'Day 2: Tabular'),
    (4.0, 0.8, 'Day 4: Policy Grad'),
    (7.5, 0.8, 'Day 5: MARL'),
]:
    ax.text(xp, yp, txt, ha='center', fontsize=8.5, color='#37474F',
            style='italic')

ax.text(5, 0.3, '— Dr. Yaé Ulrich Gaba   |   AIRINA Labs / ACAS —',
        ha='center', fontsize=9, color='#546E7A', style='italic')

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------------
# Closing remarks — print a structured summary
# ---------------------------------------------------------------

summary = """
╔══════════════════════════════════════════════════════════════════════╗
║          REINFORCEMENT LEARNING: THEORY TO PRACTICE                 ║
║          5-Day Workshop Summary — Dr. Yaé Ulrich Gaba               ║
║          AIRINA Labs / ACAS                                          ║
╠══════════════════════════════════════════════════════════════════════╣
║  Day 1 │ MDP foundations, Bellman equations, DP                     ║
║  Day 2 │ Q-Learning, SARSA, MC, eligibility traces                  ║
║  Day 3 │ DQN, experience replay, target networks, extensions        ║
║  Day 4 │ REINFORCE, A2C, PPO — policy gradient methods              ║
║  Day 5 │ MARL, self-play, Dyna-Q, behavioral cloning, capstone      ║
╠══════════════════════════════════════════════════════════════════════╣
║  Capstone: DQN for African Microgrid Energy Management              ║
║    → Learned to maximise solar use, avoid blackouts, cut diesel     ║
╠══════════════════════════════════════════════════════════════════════╣
║  Next steps:                                                        ║
║    • Implement SAC for continuous-action environments                ║
║    • Explore MAPPO for multi-grid cooperative dispatch               ║
║    • Apply constrained RL (CMDP) for safety-critical control        ║
║    • Study Decision Transformers for offline policy distillation    ║
╚══════════════════════════════════════════════════════════════════════╝
"""
print(summary)

---

## Exercises

### Part 1 — Multi-Agent RL
1. Extend `CoopGridWorld` to 3 agents. Does IQL still converge? At what cost in terms of episodes?
2. Implement a **Centralised Critic** variant (like MADDPG): the critic has access to the joint state-action, but each actor only sees its own observation.

### Part 2 — Self-Play
3. Run RPS self-play with different learning rates. Does a higher $\alpha$ help or hurt convergence to Nash? Plot $\|p - (1/3, 1/3, 1/3)\|$ over time.
4. Implement **fictitious play** for RPS: each player best-responds to the empirical frequency of the opponent's actions.

### Part 3 — Dyna-Q
5. Test Dyna-Q on `MiniGrid` or `Taxi-v3`. What is the optimal $n$ for each environment?
6. Implement **Dyna-Q+**: add an exploration bonus $\kappa\sqrt{\tau}$ for (s, a) pairs not visited recently.

### Part 4 — Offline RL
7. Corrupt 20% of expert labels with random actions. How much does BC performance degrade?
8. Implement **DAgger** (Dataset Aggregation): ask the expert to relabel states visited by the BC clone, and retrain.

### Part 5 — Capstone Extension
9. Add a second grid zone to `AfricanMicrogridEnv` with separate demand and battery — now it is a two-agent cooperative problem. Train with IQL.
10. Replace discrete actions with continuous control (charge rate $\in [0, 1]$) and use SAC from stable-baselines3.

---

*Workshop materials by Dr. Yaé Ulrich Gaba — AIRINA Labs / ACAS. For academic and educational use.*